# 8-Adaptive Thinking

`2026-03` 起仓库**删除了**独立的 `train_reason.py` / `reason_*.pth`。思考能力下沉到模板层：

- `open_thinking=0`：默认注入空 `<think>\n\n</think>`，模型倾向直接回答
- `open_thinking=1`：先写入 `<think>`，模型继续写思考再回答
- 训练时用空 think、显式 `reasoning_content` 和 `thinking_ratio` 混合，让同一套权重既能直答也能想

旧的 `<think>…</think><answer>…</answer>` 加权惩罚方案已经过时。


In [ ]:
import os, sys, math, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
import torch
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("../model")
print("device:", device, "vocab:", tokenizer.vocab_size)


## 同一条消息，两种模板


In [ ]:
messages = [{"role": "user", "content": "1+1 等于多少？"}]
print("==== open_thinking=False ====")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, open_thinking=False))
print("==== open_thinking=True ====")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, open_thinking=True))


## 训练时的空 think 采样

`post_processing_chat`：如果模板里是空 think，默认约 80% 概率删掉，避免模型只会输出空标签。


In [ ]:
from dataset.lm_dataset import post_processing_chat
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "你好"}, {"role": "assistant", "content": "你好！"}],
    tokenize=False, add_generation_prompt=False, open_thinking=False,
)
print("raw:\n", prompt)
print("keep empty think:\n", post_processing_chat(prompt, empty_think_ratio=1.0))
print("drop empty think:\n", post_processing_chat(prompt, empty_think_ratio=0.0))


## RLAIF 里按概率打开思考

`RLAIFDataset(thinking_ratio=0.5)` 会把一半 prompt 建成 `open_thinking=True`，供 PPO/GRPO rollout。


In [ ]:
ds_on = RLAIFDataset("./toydata/rlaif_data.jsonl", tokenizer, thinking_ratio=1.0)
ds_off = RLAIFDataset("./toydata/rlaif_data.jsonl", tokenizer, thinking_ratio=0.0)
print("on:", "<think>" in ds_on[0]["prompt"], ds_on[0]["prompt"][-80:])
print("off ends with:", repr(ds_off[0]["prompt"][-60:]))


推理入口都支持该开关：`eval_llm.py --open_thinking 1`，OpenAI API 的 `extra_body.chat_template_kwargs.open_thinking`，以及 `scripts/web_demo.py`。

当前 tool call 和显式思考一起开时不太稳，因为联合蒸馏样本还少。
